# Example 8.1.3 / 2: Complementarity Problems

Companion notebook for **8.1.3 Constraint Programming and Complementarity Solvers** in the AMPL lab Documentation, section "Complementarity Problems". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Complementarity Problems

For an MPCC with linear terms, an MP-based MIP solver turns each pair into a disjunction. This
version of the steel model allows either bands or plate, never both: `0 <= Make['bands']
complements Make['plate'] >= 0`. The first solve uses the driver defaults; the second gives the
driver `cvt:bigM=1e4`, a bound on both quantities that is valid because no market exceeds 6000:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
# bands and plate cannot both be made
subject to Either: 0 <= Make['bands'] complements Make['plate'] >= 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
for opts in ["", "cvt:bigM=1e4"]:
    ampl.solve(solver="highs", highs_options=opts, verbose=False)
    print(f"'{opts}': {ampl.solve_result}")
    print(ampl.get_value("solve_message").strip())
print(ampl.var["Make"].to_dict())

'': failure
HiGHS 1.14.0: Set bounds on variables
participating in logical expressions,
or use option cvt:bigM (with caution).
See more: mp.ampl.com/modeling-numerics.html
'cvt:bigM=1e4': solved
HiGHS 1.14.0: optimal solution; objective 192000
4 simplex iterations
1 branching nodes
{'bands': 6000, 'coils': 1400, 'plate': 0}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs;
solve;
option highs_options 'cvt:bigM=1e4';
solve;
display Make;
""")
except Exception:
    pass  # AMPL has printed the error above

[WARNING] 
	Nonsquare complementarity system:
		2 complementarities including 1 equations
		4 variables


HiGHS 1.14.0: Set bounds on variables
participating in logical expressions,
or use option cvt:bigM (with caution).
See more: mp.ampl.com/modeling-numerics.html
HiGHS 1.14.0:   cvt:bigM = 10000
HiGHS 1.14.0: optimal solution; objective 192000
4 simplex iterations
1 branching nodes
Make [*] :=
bands  6000
coils  1400
plate     0
;

